# Commodity Supply-Chain Map — Chokepoints, Routes, and Vessels

The commodity desk needs one live view: the three straits that matter,
the main routes bending through them, and every vessel currently
in-chokepoint with its cargo, speed, and destination.

This notebook builds exactly that — three layered GeoJSON
`FeatureCollection`s a frontend map consumes straight:

- **`chokepoints.geojson`** — Strait of Hormuz, Suez, Panama as
  polygons with throughput and risk metadata.
- **`routes.geojson`** — eight major commercial lanes running through
  them.
- **`vessels.geojson`** — ~40 AIS-style vessel positions with MMSI,
  DWT, cargo, speed, destination.

> **Demo scope.** AIS positions in this notebook are a hand-built
> snapshot representative of a normal day in each chokepoint. In
> production, swap the inline vessel table for a live AIS feed
> (Spire / MarineTraffic / VesselFinder) via the same Sedona DataFrame
> shape — routes, chokepoints, and the downstream GeoJSON pipeline are
> unchanged.

## 1. Setup

In [ ]:
from sedona.spark import *
import pyspark.sql.functions as f
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)
import json

config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

## 2. Chokepoints

Three polygons covering the transit corridor — tight enough to capture
only vessels in the strait itself, loose enough to catch the queuing
area on either approach. Throughput figures are public EIA / SCA /
ACP 2024 disclosures.

In [ ]:
# (id, name, commodity_focus, daily_throughput_mbd_eq, risk_note, min_lon, min_lat, max_lon, max_lat)
chokepoints = [
    ("CP-HORMUZ", "Strait of Hormuz",
     "Crude oil, LPG, LNG (Gulf exports)",
     21.0, "Iran-linked threat escalation; any closure = ~20% of global seaborne crude",
     55.8, 26.0, 57.2, 27.1),
    ("CP-SUEZ", "Suez Canal",
     "Containers, clean tankers, LNG (Europe-Asia)",
     9.5, "Red Sea / Bab el-Mandeb attacks divert to Cape of Good Hope (+10-14 days)",
     32.25, 29.90, 32.65, 31.30),
    ("CP-PANAMA", "Panama Canal",
     "LNG, grains, containers (US Gulf / East Coast ↔ Asia)",
     4.5, "Drought-driven draft restrictions cut neopanamax slots (2023-2024)",
     -80.00, 8.85, -79.45, 9.38),
]

choke_schema = StructType([
    StructField("choke_id",                 StringType()),
    StructField("name",                     StringType()),
    StructField("commodity_focus",          StringType()),
    StructField("daily_throughput_mbd_eq",  DoubleType()),
    StructField("risk_note",                StringType()),
    StructField("min_lon",                  DoubleType()),
    StructField("min_lat",                  DoubleType()),
    StructField("max_lon",                  DoubleType()),
    StructField("max_lat",                  DoubleType()),
])

chokepoints_df = sedona.createDataFrame(chokepoints, choke_schema) \
    .withColumn(
        "geometry",
        f.expr("ST_MakeEnvelope(min_lon, min_lat, max_lon, max_lat, 4326)")
    )
chokepoints_df.createOrReplaceTempView("chokepoints")
chokepoints_df.select("choke_id", "name", "commodity_focus",
                      "daily_throughput_mbd_eq").show(truncate=False)

## 3. Major Shipping Routes

Eight commercial lanes, each routed through one or more chokepoints.
LineString geometries are hand-anchored through realistic waypoints.

In [ ]:
# (route_id, name, commodity, chokepoints, wkt)
routes = [
    ("RT-01", "Ras Tanura → Singapore",   "Crude oil (VLCC)", "Hormuz",
     "LINESTRING (50.16 26.64, 56.27 26.55, 57.10 25.47, "
     "60.00 23.50, 72.00 13.00, 95.00 5.50, 103.85 1.29)"),
    ("RT-02", "Ras Tanura → Rotterdam",   "Crude oil (VLCC)", "Hormuz, Suez",
     "LINESTRING (50.16 26.64, 56.27 26.55, 57.10 25.47, 55.00 13.00, "
     "45.00 11.90, 41.00 15.50, 35.00 24.00, 32.55 29.95, "
     "32.35 31.25, 20.00 34.00, 0.00 36.00, -5.00 43.00, 4.47 51.92)"),
    ("RT-03", "Shanghai → Rotterdam",     "Containers",       "Suez",
     "LINESTRING (121.47 31.23, 114.16 22.28, 103.85 1.29, "
     "80.00 6.00, 60.00 12.00, 45.00 11.90, 41.00 15.50, "
     "35.00 24.00, 32.55 29.95, 32.35 31.25, 14.27 35.89, "
     "-5.00 43.00, 4.47 51.92)"),
    ("RT-04", "Shanghai → Los Angeles",   "Containers",       "—",
     "LINESTRING (121.47 31.23, 140.00 35.00, 170.00 40.00, "
     "-170.00 38.00, -140.00 35.00, -118.27 33.72)"),
    ("RT-05", "US Gulf → Yokohama",       "LNG",              "Panama",
     "LINESTRING (-95.31 29.00, -86.00 25.00, -81.00 15.00, "
     "-79.85 9.35, -79.75 8.90, -79.00 8.00, -95.00 10.00, "
     "-140.00 20.00, 160.00 30.00, 139.63 35.47)"),
    ("RT-06", "US East Coast → Japan",    "Grains (Panamax)", "Panama",
     "LINESTRING (-90.07 29.95, -83.00 25.00, -81.00 15.00, "
     "-79.85 9.35, -79.75 8.90, -79.00 8.00, -100.00 15.00, "
     "-140.00 25.00, 160.00 30.00, 139.63 35.47)"),
    ("RT-07", "Saudi → India (West)",     "Crude oil (Suezmax)", "Hormuz",
     "LINESTRING (50.16 26.64, 56.27 26.55, 57.10 25.47, "
     "63.00 23.00, 70.00 19.50, 72.83 18.94)"),
    ("RT-08", "Qatar → South Korea",      "LNG",              "Hormuz",
     "LINESTRING (51.53 25.29, 56.27 26.55, 57.10 25.47, "
     "62.00 22.00, 80.00 7.00, 100.00 3.00, 120.00 20.00, 126.58 37.54)"),
]

route_schema = StructType([
    StructField("route_id",    StringType()),
    StructField("name",        StringType()),
    StructField("commodity",   StringType()),
    StructField("chokepoints", StringType()),
    StructField("wkt",         StringType()),
])

routes_df = sedona.createDataFrame(routes, route_schema) \
    .withColumn("geometry", f.expr("ST_SetSRID(ST_GeomFromText(wkt), 4326)"))
routes_df.createOrReplaceTempView("routes")

print(f"Routes defined: {routes_df.count()}")
routes_df.select("route_id", "name", "commodity", "chokepoints") \
         .show(truncate=False)

## 4. AIS Vessel Positions (Snapshot)

Forty vessel positions representative of a normal operating day —
distributed across the three chokepoints and en-route segments.
Attributes mirror an AIS feed: MMSI, IMO, type, DWT, speed, heading,
destination ETA.

In [ ]:
# (mmsi, vessel_name, vessel_type, flag, dwt_t, lon, lat, speed_kts, heading_deg, destination, cargo)
vessels = [
    # --- Hormuz (12 vessels) ---
    (470567000, "Front Alfa",         "VLCC",          "MH", 298000, 56.35, 26.50,  12.4,  118, "SINGAPORE", "Crude Oil"),
    (431892000, "Diamond Star",       "VLCC",          "JP", 306000, 56.90, 26.38,  13.1,  112, "CHIBA",     "Crude Oil"),
    (311012300, "Olympic Bravery",    "Suezmax",       "BS", 157000, 56.12, 26.61,  11.8,  120, "MUMBAI",    "Crude Oil"),
    (636019800, "Al Rumaila",         "VLCC",          "LR", 311000, 56.55, 26.47,  12.2,  115, "YOKOHAMA",  "Crude Oil"),
    (355091000, "Pacific Venus",      "LR2",           "PA",  96000, 56.70, 26.52,  13.4,  110, "ROTTERDAM", "Diesel"),
    (538009000, "Bunga Kasturi",      "VLCC",          "MY", 300000, 56.25, 26.55,   4.2,  295, "RAS TANURA", "Ballast"),
    (403512000, "Meymandi",           "Suezmax",       "IR", 165000, 56.45, 26.70,   3.8,  290, "KHARG ISL", "Ballast"),
    (235108900, "British Sapphire",   "VLCC",          "GB", 301000, 56.95, 26.40,  12.6,  114, "ULSAN",     "Crude Oil"),
    (441234000, "Aquila LNG",         "LNG Carrier",   "KR", 170000, 56.63, 26.48,  14.2,  117, "INCHEON",   "LNG"),
    (538002100, "MOL Courage",        "LNG Carrier",   "MH", 165000, 56.82, 26.43,  13.9,  115, "YOKOHAMA",  "LNG"),
    (518900500, "Aries Sun",          "Aframax",       "CY", 115000, 56.18, 26.58,  11.1,  120, "KANDLA",    "Crude Oil"),
    (248887000, "Aegean Dream",       "VLCC",          "MT", 299000, 57.05, 26.52,   5.1,  293, "FUJAIRAH",  "Ballast"),
    # --- Suez (10 vessels) ---
    (636018210, "MSC Gülsün",         "Container",     "LR", 225000, 32.45, 30.80,  11.5,    5, "ROTTERDAM", "Containers"),
    (235102800, "Ever Given",         "Container",     "PA", 220940, 32.55, 31.10,  10.8,    8, "ROTTERDAM", "Containers"),
    (477990400, "CMA CGM Bougainville","Container",    "FR", 200000, 32.40, 30.20,  10.2,    3, "HAMBURG",   "Containers"),
    (538003070, "MOL Triumph",        "Container",     "MH", 190000, 32.52, 31.20,  11.9,  185, "SHANGHAI",  "Containers"),
    (357012800, "Maersk Hamburg",     "Container",     "DK", 141000, 32.42, 30.40,  10.5,    2, "ROTTERDAM", "Containers"),
    (357123000, "Stena Polaris",      "Suezmax",       "BM", 158000, 32.47, 31.05,   9.8,    6, "GDANSK",    "Crude Oil"),
    (538990211, "Al Safliya",         "LNG Carrier",   "QA", 152000, 32.43, 30.55,  11.1,    4, "MONTOIR",   "LNG"),
    (477993300, "COSCO Shipping Sirius","Container",   "CN", 180000, 32.38, 30.10,  10.0,  355, "PIRAEUS",   "Containers"),
    (265773000, "Stena Forerunner",   "LR1",           "SE",  74000, 32.50, 31.15,  11.6,    7, "HAMBURG",   "Jet Fuel"),
    (319019300, "NYK Romulus",        "Container",     "KY", 155000, 32.42, 30.00,   2.1,  355, "ROTTERDAM", "Containers"),
    # --- Panama (8 vessels) ---
    (311020100, "Creole Spirit",      "LNG Carrier",   "BS", 170000, -79.75, 9.15,   9.8,  280, "TOKYO",     "LNG"),
    (477881100, "Golar Snow",         "LNG Carrier",   "HK", 172000, -79.85, 9.10,   9.5,  275, "TAICHUNG",  "LNG"),
    (538001200, "Starboard Bay",      "Panamax Bulker","MH",  82000, -79.70, 9.22,   8.9,  285, "OSAKA",     "Corn"),
    (371889000, "Baltic Tern",        "Panamax Bulker","PA",  80000, -79.80, 9.05,   9.0,  280, "KAOHSIUNG", "Soybeans"),
    (244750900, "ZIM San Francisco",  "Neopanamax",    "NL", 108000, -79.88, 9.18,   9.3,  278, "YOKOHAMA",  "Containers"),
    (477557300, "LNG Schneeweisschen","LNG Carrier",   "HK", 162000, -79.60, 9.30,   2.3,   90, "SABINE PASS", "Ballast"),
    (215019300, "CMA CGM Libra",      "Neopanamax",    "MT",  96000, -79.92, 9.08,   8.8,  275, "BUSAN",     "Containers"),
    (357145000, "Cap San Antonio",    "Panamax Bulker","PA",  82000, -79.75, 9.00,   1.8,   85, "NEW ORLEANS", "Ballast"),
    # --- In transit, outside chokepoint polygons (10 vessels) ---
    (565778900, "Pacific Horizon",    "Container",     "SG", 195000, 140.50, 35.10, 21.4,   85, "LOS ANGELES", "Containers"),
    (212009800, "MSC Jewel",          "Container",     "CY", 175000, -165.00, 39.50, 22.1,   80, "LOS ANGELES", "Containers"),
    (413987000, "COSCO Galaxy",       "Container",     "CN", 210000, 103.85, 1.29,  12.7,  280, "ROTTERDAM",   "Containers"),
    (248014300, "Maran Cassiopeia",   "VLCC",          "GR", 320000, 72.00, 13.00,  13.2,  260, "ROTTERDAM",   "Crude Oil"),
    (477200500, "APL Mexico City",    "Container",     "SG", 114000, 160.00, 30.00, 20.5,   95, "LOS ANGELES", "Containers"),
    (636016500, "NCC Shams",          "MR",            "LR",  46000, 80.00, 7.00,   13.8,   75, "SINGAPORE",   "Gasoil"),
    (211019200, "Thor Independence",  "Capesize Bulker","DE",180000, 21.00, -35.00, 13.1,  140, "QINGDAO",     "Iron Ore"),
    (316043000, "Anna Oldendorff",    "Supramax Bulker","CA", 58000, -45.00, 22.00, 12.5,  110, "MUMBAI",      "Coal"),
    (244740890, "Eagle Louisiana",    "LR2",           "NL",  96000, 45.00, 11.90,  12.1,  270, "ROTTERDAM",   "Diesel"),
    (538090010, "Grand Aniva",        "LNG Carrier",   "MH", 150000, 160.00, 30.00, 15.2,  265, "TOKYO",       "LNG"),
]

vessel_schema = StructType([
    StructField("mmsi",        IntegerType()),
    StructField("vessel_name", StringType()),
    StructField("vessel_type", StringType()),
    StructField("flag",        StringType()),
    StructField("dwt_t",       IntegerType()),
    StructField("lon",         DoubleType()),
    StructField("lat",         DoubleType()),
    StructField("speed_kts",   DoubleType()),
    StructField("heading_deg", IntegerType()),
    StructField("destination", StringType()),
    StructField("cargo",       StringType()),
])

vessels_df = sedona.createDataFrame(vessels, vessel_schema) \
    .withColumn("geometry",
                f.expr("ST_SetSRID(ST_Point(lon, lat), 4326)"))
vessels_df.createOrReplaceTempView("vessels")

print(f"Vessel positions loaded: {vessels_df.count()}")
vessels_df.groupBy("vessel_type").count() \
          .orderBy(f.col("count").desc()).show(truncate=False)

## 5. Who's in Each Chokepoint — the Desk's Live Read

Point-in-polygon spatial join tags each vessel with its current
chokepoint (or `OPEN_WATER`). Group to get the live roll-up the trader
scans.

In [ ]:
tagged_vessels_df = sedona.sql("""
    SELECT
        v.*,
        COALESCE(c.choke_id, 'OPEN_WATER') AS choke_id,
        COALESCE(c.name,     'Open water') AS choke_name,
        CASE
            WHEN v.speed_kts < 5.0 THEN 'SLOW_OR_WAITING'
            ELSE                        'TRANSITING'
        END AS transit_status
    FROM vessels v
    LEFT JOIN chokepoints c
      ON ST_Contains(c.geometry, v.geometry)
""").cache()
tagged_vessels_df.createOrReplaceTempView("tagged_vessels")

sedona.sql("""
    SELECT
        choke_name,
        COUNT(*)                               AS vessel_count,
        COUNT(DISTINCT vessel_type)            AS type_variety,
        ROUND(SUM(dwt_t) / 1000.0, 0)          AS total_dwt_kt,
        ROUND(AVG(speed_kts), 1)               AS avg_speed_kts,
        SUM(CASE WHEN transit_status = 'SLOW_OR_WAITING'
                 THEN 1 ELSE 0 END)            AS slow_count
    FROM tagged_vessels
    GROUP BY choke_name
    ORDER BY vessel_count DESC
""").show(truncate=False)

# Commodity mix in each chokepoint — what's actually at risk
sedona.sql("""
    SELECT
        choke_name,
        cargo,
        COUNT(*) AS vessels,
        ROUND(SUM(dwt_t) / 1000.0, 0) AS dwt_kt
    FROM tagged_vessels
    WHERE choke_id != 'OPEN_WATER'
    GROUP BY choke_name, cargo
    ORDER BY choke_name, dwt_kt DESC
""").show(truncate=False)

## 6. Export GeoJSON — Three Layers for the Dashboard

Each layer goes to its own `FeatureCollection` so the frontend map can
style them independently (polygons shaded by risk, routes colored by
commodity, vessels sized by DWT).

In [ ]:
def df_to_fc(df, geom_col, prop_cols):
    rows = df.selectExpr(*prop_cols, f"ST_AsGeoJSON({geom_col}) AS gj").collect()
    feats = []
    for r in rows:
        d = r.asDict()
        gj = d.pop("gj")
        feats.append({
            "type": "Feature",
            "properties": {
                k: (float(v) if isinstance(v, (int, float)) else v)
                for k, v in d.items()
            },
            "geometry": json.loads(gj),
        })
    return {"type": "FeatureCollection", "features": feats}

# Chokepoints
cp_fc = df_to_fc(
    chokepoints_df,
    "geometry",
    ["choke_id", "name", "commodity_focus",
     "daily_throughput_mbd_eq", "risk_note"]
)
# Attach current vessel counts so the dashboard can badge each polygon
cp_counts = {r["choke_id"]: r["cnt"] for r in sedona.sql("""
    SELECT choke_id, COUNT(*) AS cnt FROM tagged_vessels
    WHERE choke_id != 'OPEN_WATER' GROUP BY choke_id
""").toPandas().to_dict("records")}
for feat in cp_fc["features"]:
    feat["properties"]["live_vessel_count"] = int(
        cp_counts.get(feat["properties"]["choke_id"], 0)
    )

with open("/tmp/chokepoints.geojson", "w") as fh:
    json.dump(cp_fc, fh, indent=2)
print(f"Wrote /tmp/chokepoints.geojson ({len(cp_fc['features'])} features)")

# Routes
rt_fc = df_to_fc(
    routes_df,
    "geometry",
    ["route_id", "name", "commodity", "chokepoints"]
)
with open("/tmp/routes.geojson", "w") as fh:
    json.dump(rt_fc, fh, indent=2)
print(f"Wrote /tmp/routes.geojson ({len(rt_fc['features'])} features)")

# Vessels
vs_fc = df_to_fc(
    tagged_vessels_df,
    "geometry",
    ["mmsi", "vessel_name", "vessel_type", "flag", "dwt_t",
     "speed_kts", "heading_deg", "destination", "cargo",
     "choke_id", "choke_name", "transit_status"]
)
# Cast mmsi/dwt back to int in properties (they come through as float via the helper)
for feat in vs_fc["features"]:
    feat["properties"]["mmsi"]  = int(feat["properties"]["mmsi"])
    feat["properties"]["dwt_t"] = int(feat["properties"]["dwt_t"])

with open("/tmp/vessels.geojson", "w") as fh:
    json.dump(vs_fc, fh, indent=2)
print(f"Wrote /tmp/vessels.geojson ({len(vs_fc['features'])} features)")

## 7. Preview on a Map

In [ ]:
viz = SedonaKepler.create_map()
SedonaKepler.add_df(viz,
    chokepoints_df.select("choke_id", "name",
                          "daily_throughput_mbd_eq", "geometry"),
    name="Chokepoints")
SedonaKepler.add_df(viz,
    routes_df.select("route_id", "name", "commodity", "geometry"),
    name="Shipping Routes")
SedonaKepler.add_df(viz,
    tagged_vessels_df.select(
        "vessel_name", "vessel_type", "cargo", "speed_kts",
        "choke_name", "transit_status", "geometry"
    ),
    name="Vessels (AIS)")
viz

---

## Outputs

| File | Contents |
|---|---|
| `/tmp/chokepoints.geojson` | Polygons for Hormuz / Suez / Panama, throughput + live vessel count per polygon |
| `/tmp/routes.geojson` | Eight LineString commercial lanes with commodity + chokepoint list |
| `/tmp/vessels.geojson` | Point features: MMSI, type, DWT, speed, destination, cargo, transit status |

## Going live

| Layer | Swap for |
|---|---|
| Chokepoints | Same polygons; may refine from UNCTAD / IMO corridor definitions |
| Routes | MarineTraffic density-derived routes or AIS-trace clustering |
| Vessels | Live AIS feed (Spire / MarineTraffic / VesselFinder) — Kafka → Sedona streaming → re-publish the GeoJSON every 5 min |